# ДЗ 3. Ускорение Python: Cython, Numba, ctypes, pybind11

Тема про то, что делать, когда чистый Python (даже с NumPy) оказывается недостаточно быстрым, а
переписывать всё на C++ целиком не хочется. Четыре инструмента, которые сегодня разберём, решают
одну и ту же задачу — вызвать скомпилированный код из Python — но очень по-разному:

- **Numba** — JIT-компиляция прямо из Python-функции (`@njit`), почти без изменений кода.
- **Cython** — Python-подобный язык, компилируется в C; чем больше типов вы укажете, тем быстрее.
- **ctypes** — вызов уже существующей скомпилированной C-библиотеки без каких-либо оберток.
- **pybind11** — написание полноценного C++ расширения с автоматической генерацией Python-биндингов.

Сквозные примеры: **множество Мандельброта** (красиво, embarrassingly parallel) и
**N-тел** — расчёт гравитационных сил между N частицами (классическая O(N²) задача, физически
осмысленная, с естественной проверкой правильности через третий закон Ньютона).

**Важно про среду выполнения.** В Google Colab доступен компилятор `g++`/`gcc` "из коробки", и
`pip install cython pybind11` работает без проблем. Если что-то из этого недоступно в вашей
среде — используйте Colab специально для этого ДЗ.

In [ ]:
# =========================
# SETUP / ENVIRONMENT CHECK
# =========================
# Запустите эту ячейку первой. Затем можно использовать Runtime -> Run all.

import importlib.util
import shutil
import subprocess
import sys


def ensure_package(module_name, pip_name=None):
    if importlib.util.find_spec(module_name) is None:
        package = pip_name or module_name
        print(f"Устанавливаем {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


ensure_package("numpy")
ensure_package("numba")
ensure_package("Cython", "cython")
ensure_package("pybind11")

missing = [name for name in ("gcc", "g++") if shutil.which(name) is None]
if missing:
    raise RuntimeError("Не найдены компиляторы: " + ", ".join(missing))

import numpy as np
import time
import matplotlib.pyplot as plt
import numba
import Cython
import pybind11

print("✓ Python:", sys.version.split()[0])
print("✓ NumPy:", np.__version__)
print("✓ Numba:", numba.__version__)
print("✓ Cython:", Cython.__version__)
print("✓ pybind11:", pybind11.__version__)
print("✓ gcc:", shutil.which("gcc"))
print("✓ g++:", shutil.which("g++"))
print("\nСреда готова к выполнению ДЗ 3.")

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

## Уровень 1. Попроще

### Задача 1.1. Множество Мандельброта: Python vs NumPy vs Numba

Точка $c$ принадлежит множеству Мандельброта, если последовательность $z_{n+1} = z_n^2 + c$
(с $z_0=0$) не уходит на бесконечность. На практике: если $|z_n| > 2$, последовательность гарантированно
разойдётся — тогда запоминаем номер итерации `it`, на которой это произошло (чем меньше — тем
"дальше" точка от множества).

1. Реализуйте `mandelbrot_python(width, height, max_iter, ...)` — наивная версия с вложенными
   Python-циклами по пикселям.
2. Реализуйте `mandelbrot_numba` — **тот же самый код**, обёрнутый в `@njit`.
3. Реализуйте `mandelbrot_numpy` — векторизованная версия без явных циклов по пикселям (все точки
   обрабатываются одновременно на каждой итерации, с помощью булевой маски "ещё не разошедшихся"
   точек).
4. Сравните все три на **точное** совпадение результата (не приближённое!) и на скорость.
   Визуализируйте результат через `plt.imshow`.

**Подсказка/ловушка:** используйте `np.linspace` для сетки координат **одинаково** во всех трёх
версиях — на первый взгляд эквивалентные формулы сетки (`xmin + j*(xmax-xmin)/width` против
`np.linspace(xmin, xmax, width)`) на самом деле немного отличаются (разный шаг: делится на
`width` или на `width-1`), и результаты не совпадут "неожиданно" по причинам, не имеющим отношения
к Cython/Numba.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from numba import njit


def _validate_mandelbrot_args(width, height, max_iter, xmin, xmax, ymin, ymax):
    if not isinstance(width, (int, np.integer)) or width < 1:
        raise ValueError("width must be a positive integer")
    if not isinstance(height, (int, np.integer)) or height < 1:
        raise ValueError("height must be a positive integer")
    if not isinstance(max_iter, (int, np.integer)) or max_iter < 1:
        raise ValueError("max_iter must be a positive integer")
    if not all(np.isfinite(v) for v in (xmin, xmax, ymin, ymax)):
        raise ValueError("bounds must be finite")
    if xmin >= xmax or ymin >= ymax:
        raise ValueError("bounds must satisfy min < max")


def mandelbrot_python(width, height, max_iter, xmin=-2.0, xmax=1.0, ymin=-1.5, ymax=1.5):
    _validate_mandelbrot_args(width, height, max_iter, xmin, xmax, ymin, ymax)
    xs = np.linspace(xmin, xmax, width, dtype=np.float64)
    ys = np.linspace(ymin, ymax, height, dtype=np.float64)
    result = np.full((height, width), max_iter, dtype=np.int32)

    for iy, y in enumerate(ys):
        for ix, x in enumerate(xs):
            c = complex(x, y)
            z = 0.0j
            for it in range(max_iter):
                z = z * z + c
                if (z.real * z.real + z.imag * z.imag) > 4.0:
                    result[iy, ix] = it
                    break
    return result


@njit
def mandelbrot_numba(width, height, max_iter, xmin=-2.0, xmax=1.0, ymin=-1.5, ymax=1.5):
    # Параметры валидируются в Python-wrapper ниже; здесь только вычислительное ядро.
    xs = np.linspace(xmin, xmax, width)
    ys = np.linspace(ymin, ymax, height)
    result = np.full((height, width), max_iter, dtype=np.int32)

    for iy in range(height):
        y = ys[iy]
        for ix in range(width):
            x = xs[ix]
            zr = 0.0
            zi = 0.0
            for it in range(max_iter):
                new_zr = zr * zr - zi * zi + x
                new_zi = 2.0 * zr * zi + y
                zr = new_zr
                zi = new_zi
                if zr * zr + zi * zi > 4.0:
                    result[iy, ix] = it
                    break
    return result


def mandelbrot_numba_checked(width, height, max_iter, xmin=-2.0, xmax=1.0, ymin=-1.5, ymax=1.5):
    _validate_mandelbrot_args(width, height, max_iter, xmin, xmax, ymin, ymax)
    return mandelbrot_numba(width, height, max_iter, xmin, xmax, ymin, ymax)


def mandelbrot_numpy(width, height, max_iter, xmin=-2.0, xmax=1.0, ymin=-1.5, ymax=1.5):
    _validate_mandelbrot_args(width, height, max_iter, xmin, xmax, ymin, ymax)
    xs = np.linspace(xmin, xmax, width, dtype=np.float64)
    ys = np.linspace(ymin, ymax, height, dtype=np.float64)
    c = xs[None, :] + 1j * ys[:, None]
    z = np.zeros_like(c, dtype=np.complex128)
    active = np.ones(c.shape, dtype=bool)
    result = np.full(c.shape, max_iter, dtype=np.int32)

    for it in range(max_iter):
        z[active] = z[active] * z[active] + c[active]
        escaped = active & ((z.real * z.real + z.imag * z.imag) > 4.0)
        result[escaped] = it
        active[escaped] = False
        if not active.any():
            break
    return result


# Одинаковый тестовый набор для всех трёх реализаций.
test_args = dict(width=320, height=240, max_iter=80)
r_python = mandelbrot_python(**test_args)
r_numba = mandelbrot_numba_checked(**test_args)
r_numpy = mandelbrot_numpy(**test_args)

assert np.array_equal(r_python, r_numba)
assert np.array_equal(r_python, r_numpy)
assert np.all((r_python >= 0) & (r_python <= test_args["max_iter"]))
print("✓ Python == Numba: True")
print("✓ Python == NumPy: True")
print("✓ Три реализации дают точно одинаковый результат.")

# Warm-up: компиляция Numba происходит здесь, до benchmark.
mandelbrot_numba_checked(**test_args)
assert mandelbrot_numba.nopython_signatures

benchmark_args = dict(width=500, height=500, max_iter=100)

def benchmark(fn, repeats=5):
    samples = []
    value = None
    for _ in range(repeats):
        start = time.perf_counter()
        value = fn(**benchmark_args)
        samples.append(time.perf_counter() - start)
    samples = np.asarray(samples, dtype=np.float64)
    median = float(np.median(samples))
    assert np.all(np.isfinite(samples)) and np.all(samples > 0)
    assert np.isfinite(median) and median > 0
    return value, samples, median

bench = {
    "Python": benchmark(mandelbrot_python),
    "NumPy": benchmark(mandelbrot_numpy),
    "Numba": benchmark(mandelbrot_numba_checked),
}

# Отдельно проверяем корректность именно тех результатов, которые были измерены.
assert np.array_equal(bench["Python"][0], bench["NumPy"][0])
assert np.array_equal(bench["Python"][0], bench["Numba"][0])

for name, (_, samples, median) in bench.items():
    print(f"{name:>6}: median={median:.6f} s | samples={np.round(samples, 6)}")

python_time = bench["Python"][2]
numpy_time = bench["NumPy"][2]
numba_time = bench["Numba"][2]
print(f"\nNumPy speedup vs Python: {python_time / numpy_time:.2f}x")
print(f"Numba speedup vs Python: {python_time / numba_time:.2f}x")
print(f"Numba speedup vs NumPy:  {numpy_time / numba_time:.2f}x")
print("✓ Все benchmark-времена положительные и конечные.")
print("✓ Результаты benchmark-вычислений точно совпадают.")
print("✓ Benchmark выполнен после warm-up Numba.")

plt.figure(figsize=(7, 5))
plt.imshow(bench["Python"][0], cmap="magma", origin="lower")
plt.title("Mandelbrot set")
plt.xlabel("x pixel")
plt.ylabel("y pixel")
plt.colorbar(label="escape iteration")
plt.tight_layout()
plt.show()
print("✓ Результат визуализирован.")

In [ ]:
# Corner cases для общей логики Mandelbrot.
for fn in (mandelbrot_python, mandelbrot_numpy, mandelbrot_numba_checked):
    assert fn(1, 1, 1).shape == (1, 1)

bad_cases = [
    dict(width=0, height=10, max_iter=10),
    dict(width=10, height=0, max_iter=10),
    dict(width=10, height=10, max_iter=0),
    dict(width=10, height=10, max_iter=10, xmin=1.0, xmax=1.0),
    dict(width=10, height=10, max_iter=10, xmin=np.nan),
]
for bad in bad_cases:
    for fn in (mandelbrot_python, mandelbrot_numpy, mandelbrot_numba_checked):
        try:
            fn(**bad)
        except (ValueError, TypeError):
            pass
        else:
            raise AssertionError(f"{fn.__name__}: invalid parameters were accepted")
print("✓ Corner cases Mandelbrot проверены.")

### Задача 1.2. Первый Cython-модуль: важность типизации

Возьмите численное интегрирование методом трапеций
$\int_a^b f(x)\,dx \approx h\left(\tfrac12 f(a) + \tfrac12 f(b) + \sum_{i=1}^{n-1} f(a+ih)\right)$
для $f(x) = x^4 - 3x^3 + 2$ на большом числе точек ($n \sim 2 \cdot 10^6$).

1. Реализуйте `integrate_trapz_python` — обычный Python-цикл, принимающий `f` как аргумент.
2. В ячейке `%%cython` реализуйте `integrate_trapz_cython_untyped` — **тот же самый код**, просто
   скопированный в cython-ячейку, без единой типизации (`f` по-прежнему обычная Python-функция).
3. В другой `%%cython`-ячейке реализуйте `integrate_trapz_cython_full` — версия, где $f$
   "зашита" как `cdef double f_c(double x)` прямо внутри модуля (не передаётся как аргумент), а
   все переменные цикла типизированы (`cdef double`, `cdef long`).
4. Сравните время всех трёх версий. Объясните, почему шаг 2 почти не даёт ускорения, а шаг 3 —
   даёт заметное.

In [ ]:
# Обычный Python-цикл для интегрирования методом трапеций.

def f(x):
    return x**4 - 3*x**3 + 2


def integrate_trapz_python(a, b, n, f):
    if not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError("n must be >= 1")
    if not (np.isfinite(a) and np.isfinite(b)):
        raise ValueError("a and b must be finite")
    h = (b - a) / n
    total = 0.5 * f(a) + 0.5 * f(b)
    for i in range(1, n):
        total += f(a + i * h)
    return h * total


def exact_integral(a, b):
    def F(x):
        return x**5 / 5.0 - 3.0 * x**4 / 4.0 + 2.0 * x
    return F(b) - F(a)

small = integrate_trapz_python(-1.0, 2.0, 10_000, f)
assert np.isclose(small, exact_integral(-1.0, 2.0), rtol=1e-7, atol=1e-9)
print("Python-реализация корректна.")

In [ ]:
# Подключаем Cython один раз, без служебного сообщения при повторном запуске.
ip = get_ipython()
if "Cython" not in ip.extension_manager.loaded:
    ip.run_line_magic("load_ext", "Cython")
print("✓ Cython extension готов")


In [ ]:
%%cython
# Намеренно нетипизированная версия: аргументы и переменные остаются Python-объектами.
def integrate_trapz_cython_untyped(a, b, n, f):
    if n < 1:
        raise ValueError("n must be >= 1")
    h = (b - a) / n
    total = 0.5 * f(a) + 0.5 * f(b)
    for i in range(1, n):
        total += f(a + i * h)
    return h * total

In [ ]:
%%cython
# Полностью типизированная вычислительная версия.
cdef inline double f_c(double x):
    return x**4 - 3.0*x**3 + 2.0

def integrate_trapz_cython_full(double a, double b, long n):
    cdef double h, total, x
    cdef long i
    if n < 1:
        raise ValueError("n must be >= 1")
    h = (b - a) / n
    total = 0.5 * f_c(a) + 0.5 * f_c(b)
    for i in range(1, n):
        x = a + i * h
        total += f_c(x)
    return h * total

In [ ]:
import time

a, b = -1.0, 2.0
n = 2_000_000

# Санити-проверка до большого benchmark.
r_py = integrate_trapz_python(a, b, 10_000, f)
r_untyped = integrate_trapz_cython_untyped(a, b, 10_000, f)
r_full = integrate_trapz_cython_full(a, b, 10_000)
assert np.isclose(r_py, r_untyped, rtol=1e-12, atol=1e-12)
assert np.isclose(r_py, r_full, rtol=1e-12, atol=1e-12)

# Проверяем сходимость метода трапеций отдельно.
exact = exact_integral(a, b)
e1 = abs(integrate_trapz_python(a, b, 10_000, f) - exact)
e2 = abs(integrate_trapz_python(a, b, 20_000, f) - exact)
print(f"Точное значение: {exact:.12f}")
print(f"Ошибка n=10000: {e1:.3e}")
print(f"Ошибка n=20000: {e2:.3e}")
assert e2 < e1
assert e2 / e1 < 0.30  # для O(h^2) ожидаем примерно 1/4


def measure(fn, repeats=3):
    samples=[]
    value=None
    for _ in range(repeats):
        t0=time.perf_counter(); value=fn(); samples.append(time.perf_counter()-t0)
    samples=np.asarray(samples)
    assert np.all(np.isfinite(samples)) and np.all(samples > 0)
    return value, samples, float(np.median(samples))

results = {
    "Python": measure(lambda: integrate_trapz_python(a,b,n,f)),
    "Cython untyped": measure(lambda: integrate_trapz_cython_untyped(a,b,n,f)),
    "Cython full": measure(lambda: integrate_trapz_cython_full(a,b,n)),
}

for name, (value, samples, median) in results.items():
    print(f"{name:>15}: median={median:.6f} s | samples={np.round(samples,6)} | value={value:.12f}")

assert np.isclose(results["Python"][0], results["Cython untyped"][0], rtol=1e-10, atol=1e-10)
assert np.isclose(results["Python"][0], results["Cython full"][0], rtol=1e-10, atol=1e-10)

print("\nПочему untyped Cython почти не ускоряет: внутри цикла остаются Python-объекты и вызов Python-функции f.")
print("Почему full Cython быстрее: арифметика, цикл и f_c выполняются как типизированный C-код без Python-overhead.")

### Задача 1.3. Своя C-функция через ctypes

`ctypes` — самый "низкоуровневый" из четырёх инструментов: он не генерирует и не компилирует
ничего сам, а просто вызывает уже скомпилированную библиотеку (`.so`/`.dll`) напрямую из Python.

1. Напишите на C функцию `moving_stats`, которая одним проходом по массиву считает и среднее, и
   стандартное отклонение (через указатели `out_mean`, `out_std` для двух выходных значений).
2. Скомпилируйте её в динамическую библиотеку: `gcc -O3 -shared -fPIC -o mylib.so mylib.c -lm`.
3. В Python через `ctypes.CDLL` загрузите библиотеку, объявите `argtypes`/`restype` для функции
   (это обязательно — без явного объявления типов `ctypes` не знает, как передавать `double*`
   и не защищён от падения по неверной сигнатуре) и передайте `numpy`-массив через
   `array.ctypes.data_as(...)` (без копирования!).
4. Сравните скорость с (а) чистым Python-циклом, (б) `numpy.mean()` + `numpy.std()`. Объясните
   результат.

In [ ]:
# Записываем исходник C без вывода служебного сообщения от Jupyter.
c_source = r"""
#include <math.h>
#include <stddef.h>

/* One-pass mean + population standard deviation (Welford). */
int moving_stats(const double* data, int n, double* out_mean, double* out_std) {
    if (data == NULL || out_mean == NULL || out_std == NULL || n <= 0) return -1;
    double mean = 0.0;
    double m2 = 0.0;
    for (int i = 0; i < n; ++i) {
        double x = data[i];
        double delta = x - mean;
        mean += delta / (i + 1);
        double delta2 = x - mean;
        m2 += delta * delta2;
    }
    *out_mean = mean;
    *out_std = sqrt(m2 / n);
    return 0;
}
"""
with open("mylib.c", "w", encoding="utf-8") as f:
    f.write(c_source)
print("✓ mylib.c создан")


In [ ]:
!gcc -O3 -shared -fPIC -o mylib.so mylib.c -lm

In [ ]:
import ctypes

lib = ctypes.CDLL("./mylib.so")
lib.moving_stats.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags="C_CONTIGUOUS"),
    ctypes.c_int,
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_double),
]
lib.moving_stats.restype = ctypes.c_int


def moving_stats_ctypes(array):
    array = np.asarray(array)
    if array.ndim != 1:
        raise ValueError("array must be one-dimensional")
    if array.size == 0:
        raise ValueError("array must be non-empty")
    if not np.issubdtype(array.dtype, np.number):
        raise TypeError("array must be numeric")
    if not np.all(np.isfinite(array)):
        raise ValueError("array must contain only finite values")
    array = np.ascontiguousarray(array, dtype=np.float64)
    out_mean = ctypes.c_double()
    out_std = ctypes.c_double()
    status = lib.moving_stats(array, array.size, ctypes.byref(out_mean), ctypes.byref(out_std))
    if status != 0:
        raise RuntimeError(f"moving_stats failed with status={status}")
    return out_mean.value, out_std.value


def moving_stats_python(array):
    if len(array) == 0:
        raise ValueError("array must be non-empty")
    total = 0.0
    for x in array:
        total += float(x)
    mean = total / len(array)
    sq = 0.0
    for x in array:
        d = float(x) - mean
        sq += d*d
    return mean, float(np.sqrt(sq / len(array)))

rng = np.random.default_rng(0)
data = rng.normal(size=200_000).astype(np.float64)

m_py, s_py = moving_stats_python(data)
m_c, s_c = moving_stats_ctypes(data)
m_np, s_np = float(np.mean(data)), float(np.std(data))
assert np.allclose([m_py,s_py], [m_c,s_c], rtol=1e-10, atol=1e-10)
assert np.allclose([m_np,s_np], [m_c,s_c], rtol=1e-10, atol=1e-10)


def measure(fn, repeats=5):
    times=[]; value=None
    for _ in range(repeats):
        t0=time.perf_counter(); value=fn(); times.append(time.perf_counter()-t0)
    times=np.asarray(times)
    assert np.all(np.isfinite(times)) and np.all(times > 0)
    return value, float(np.median(times))

bench_stats = {
    "Python": measure(lambda: moving_stats_python(data)),
    "ctypes + C": measure(lambda: moving_stats_ctypes(data)),
    "NumPy mean/std": measure(lambda: (float(np.mean(data)), float(np.std(data)))),
}
for name,(value,elapsed) in bench_stats.items():
    print(f"{name:>16}: median={elapsed:.6f}s, mean={value[0]:.6f}, std={value[1]:.6f}")
print("\nctypes передаёт NumPy-буфер в C без промежуточного копирования данных.")
print("NumPy остаётся конкурентоспособным, потому что mean/std уже реализованы в оптимизированном C.")

In [ ]:
for bad in (np.array([], dtype=np.float64), np.array([[1.0, 2.0]]), np.array([np.nan]), np.array([np.inf])):
    try:
        moving_stats_ctypes(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid input was accepted")

m, s = moving_stats_ctypes(np.array([7.0], dtype=np.float64))
assert m == 7.0 and s == 0.0
print("✓ Corner cases ctypes/C проверены.")

## Уровень 2. Посложнее

### Задача 2.1. N тел: ускоряем гравитационную симуляцию с Numba

Классическая задача N тел: для $N$ частиц с массами $m_i$ и позициями $\vec r_i$ нужно посчитать
силу, действующую на каждую частицу со стороны всех остальных —
$\vec F_i = \sum_{j \ne i} G \dfrac{m_i m_j}{|\vec r_j - \vec r_i|^2}\cdot \dfrac{\vec r_j - \vec
r_i}{|\vec r_j - \vec r_i|}$. Это $O(N^2)$ по построению (каждая пара взаимодействует).

1. Реализуйте `nbody_forces_python` — двойной цикл на чистом Python.
2. Реализуйте `nbody_forces_numba` — та же функция под `@njit`.
3. Реализуйте `nbody_forces_numba_parallel` — версия с `@njit(parallel=True)` и `prange` по
   внешнему циклу (по частицам `i`).
4. **Проверка корректности, а не только скорости:** по третьему закону Ньютона сумма **всех** сил
   в замкнутой системе должна быть равна нулю (силы действия и противодействия взаимно
   уничтожаются). Проверьте это как `assert` для всех трёх версий — это гораздо надёжнее, чем
   просто "код запустился и что-то вывел".

In [ ]:
from numba import njit, prange

G = 1.0
SOFTENING = 1e-6


def _validate_nbody(pos, mass):
    pos = np.asarray(pos, dtype=np.float64)
    mass = np.asarray(mass, dtype=np.float64)
    if pos.ndim != 2 or pos.shape[1] != 2:
        raise ValueError("pos must have shape (N, 2)")
    if mass.ndim != 1 or mass.shape[0] != pos.shape[0]:
        raise ValueError("mass must have shape (N,)")
    if not np.all(np.isfinite(pos)) or not np.all(np.isfinite(mass)):
        raise ValueError("pos and mass must be finite")
    if np.any(mass < 0):
        raise ValueError("mass must be non-negative")
    if pos.shape[0] == 0:
        raise ValueError("N must be > 0")
    return np.ascontiguousarray(pos), np.ascontiguousarray(mass)


def nbody_forces_python(pos, mass):
    pos, mass = _validate_nbody(pos, mass)
    n = pos.shape[0]
    forces = np.zeros((n,2), dtype=np.float64)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            dx = pos[j,0]-pos[i,0]
            dy = pos[j,1]-pos[i,1]
            dist2 = dx*dx + dy*dy + SOFTENING
            dist = np.sqrt(dist2)
            f = G*mass[i]*mass[j]/dist2
            forces[i,0] += f*dx/dist
            forces[i,1] += f*dy/dist
    return forces


@njit
def _nbody_numba_core(pos, mass):
    n=pos.shape[0]
    forces=np.zeros((n,2),dtype=np.float64)
    for i in range(n):
        for j in range(n):
            if i==j: continue
            dx=pos[j,0]-pos[i,0]
            dy=pos[j,1]-pos[i,1]
            dist2=dx*dx+dy*dy+SOFTENING
            dist=np.sqrt(dist2)
            f=G*mass[i]*mass[j]/dist2
            forces[i,0]+=f*dx/dist
            forces[i,1]+=f*dy/dist
    return forces


def nbody_forces_numba(pos, mass):
    pos,mass=_validate_nbody(pos,mass)
    return _nbody_numba_core(pos,mass)


@njit(parallel=True)
def _nbody_numba_parallel_core(pos, mass):
    n=pos.shape[0]
    forces=np.zeros((n,2),dtype=np.float64)
    for i in prange(n):
        for j in range(n):
            if i==j: continue
            dx=pos[j,0]-pos[i,0]
            dy=pos[j,1]-pos[i,1]
            dist2=dx*dx+dy*dy+SOFTENING
            dist=np.sqrt(dist2)
            f=G*mass[i]*mass[j]/dist2
            forces[i,0]+=f*dx/dist
            forces[i,1]+=f*dy/dist
    return forces


def nbody_forces_numba_parallel(pos, mass):
    pos,mass=_validate_nbody(pos,mass)
    return _nbody_numba_parallel_core(pos,mass)

rng=np.random.default_rng(1)
pos=rng.normal(size=(160,2)).astype(np.float64)
mass=rng.uniform(0.5,2.0,size=160).astype(np.float64)

f_py=nbody_forces_python(pos,mass)
f_nb=nbody_forces_numba(pos,mass)
f_par=nbody_forces_numba_parallel(pos,mass)
assert np.allclose(f_py,f_nb,rtol=1e-11,atol=1e-11)
assert np.allclose(f_py,f_par,rtol=1e-11,atol=1e-11)
for f in (f_py,f_nb,f_par):
    assert np.linalg.norm(f.sum(axis=0)) < 1e-10
print("✓ Python == Numba == Numba parallel")
print("✓ Третий закон Ньютона: сумма всех сил ≈ 0")

# Warm-up уже выполнен на корректностном тесте.
bench_pos=pos[:160]
bench_mass=mass[:160]
def measure(fn,repeats=3):
    ts=[]; value=None
    for _ in range(repeats):
        t0=time.perf_counter(); value=fn(bench_pos,bench_mass); ts.append(time.perf_counter()-t0)
    ts=np.asarray(ts)
    assert np.all(np.isfinite(ts)) and np.all(ts>0)
    return value,float(np.median(ts))

for name,fn in [("Python",nbody_forces_python),("Numba",nbody_forces_numba),("Numba parallel",nbody_forces_numba_parallel)]:
    value,t=measure(fn)
    print(f"{name:>14}: median={t:.6f}s")

### Задача 2.2. pybind11: своё C++ расширение

pybind11 — библиотека для написания Python-расширений на C++ с автоматической генерацией
биндингов (в отличие от ctypes, где вы вручную описываете сигнатуры). Реализуйте ту же задачу
N тел на C++.

1. Напишите `nbody_ext.cpp` с функцией `nbody_forces(pos, mass)`, принимающей и возвращающей
   `numpy`-массивы через `py::array_t<double>` (модуль `pybind11/numpy.h`).
2. Скомпилируйте в `.so`-модуль (команда компиляции — в решении; на Colab работает как есть).
3. Импортируйте модуль как обычный Python-модуль и сравните на корректность (третий закон
   Ньютона + совпадение с python-версией) и на скорость с Numba-версией из задачи 2.1.

In [ ]:
# Записываем исходник C++ без служебного вывода от Jupyter.
cpp_source = r"""
#include <pybind11/pybind11.h>
#include <pybind11/numpy.h>
#include <cmath>

namespace py = pybind11;

py::array_t<double> nbody_forces(py::array_t<double> pos, py::array_t<double> mass) {
    auto pos_buf = pos.unchecked<2>();
    auto mass_buf = mass.unchecked<1>();
    ssize_t n = pos_buf.shape(0);

    auto result = py::array_t<double>({n, (ssize_t)2});
    auto res_buf = result.mutable_unchecked<2>();
    const double G = 1.0;

    for (ssize_t i = 0; i < n; i++) {
        double fx = 0.0, fy = 0.0;
        for (ssize_t j = 0; j < n; j++) {
            if (i == j) continue;
            double dx = pos_buf(j,0) - pos_buf(i,0);
            double dy = pos_buf(j,1) - pos_buf(i,1);
            double dist2 = dx*dx + dy*dy + 1e-6;
            double dist = std::sqrt(dist2);
            double f = G * mass_buf(i) * mass_buf(j) / dist2;
            fx += f * dx / dist;
            fy += f * dy / dist;
        }
        res_buf(i,0) = fx;
        res_buf(i,1) = fy;
    }
    return result;
}

PYBIND11_MODULE(nbody_ext, m) {
    m.def("nbody_forces", &nbody_forces, "Compute N-body gravitational forces");
}
"""
with open("nbody_ext.cpp", "w", encoding="utf-8") as f:
    f.write(cpp_source)
print("✓ nbody_ext.cpp создан")


In [ ]:
import pybind11, sysconfig
pybind_inc=pybind11.get_include()
py_inc=sysconfig.get_path("include")
ext_suffix=sysconfig.get_config_var("EXT_SUFFIX")
assert ext_suffix
!g++ -O3 -Wall -shared -std=c++14 -fPIC -I{pybind_inc} -I{py_inc} nbody_ext.cpp -o nbody_ext{ext_suffix}
print("✓ C++ extension compiled")

In [ ]:
import importlib
nbody_ext=importlib.import_module("nbody_ext")

cpp_forces=nbody_ext.nbody_forces(pos,mass)
assert cpp_forces.shape==f_py.shape
assert np.allclose(cpp_forces,f_py,rtol=1e-11,atol=1e-11)
assert np.linalg.norm(cpp_forces.sum(axis=0)) < 1e-10
print("✓ pybind11 == Python по силам")
print("✓ Третий закон Ньютона выполнен")

def measure_cpp(repeats=3):
    ts=[]; value=None
    for _ in range(repeats):
        t0=time.perf_counter(); value=nbody_ext.nbody_forces(bench_pos,bench_mass); ts.append(time.perf_counter()-t0)
    ts=np.asarray(ts)
    assert np.all(np.isfinite(ts)) and np.all(ts>0)
    return value,float(np.median(ts))

cpp_value,cpp_time=measure_cpp()
nb_value,nb_time=measure(nbody_forces_numba)
par_value,par_time=measure(nbody_forces_numba_parallel)
assert np.allclose(cpp_value,nb_value,rtol=1e-11,atol=1e-11)
print(f"C++/pybind11: median={cpp_time:.6f}s")
print(f"Numba:        median={nb_time:.6f}s")
print(f"Numba parallel: median={par_time:.6f}s")

---
**Что сдавать:** ноутбук с выполненными измерениями и графиками, а также письменными ответами на
вопросы для решённых задач. Некоторые задачи требуют `g++`/`gcc` и `pip install cython pybind11`
— рекомендуем выполнять это ДЗ в Google Colab. Суммарное время выполнения всех ячеек — не больше
нескольких минут (основное время уходит на компиляцию C/C++/Cython-кода, а не на сами вычисления).